# Module 07 · Oscillators and the 555 timer (Floyd Ch.16)

In [1]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Frequency formulas of the RC and LC oscillators

In [2]:
print(f"Wien (6.2k, 0.02uF): {1/(2*np.pi*6.2e3*0.02e-6):.0f} Hz")
print(f"Phase-shift (4.7k, 0.022uF): {1/(2*np.pi*np.sqrt(6)*4.7e3*0.022e-6):.0f} Hz, Rf = {29*4.7:.1f} kOhm")
c1, c2, L = 470e-12, 47e-12, 1e-3; ct = c1*c2/(c1+c2)
print(f"Colpitts: CT = {ct*1e12:.1f} pF, f = {1/(2*np.pi*np.sqrt(L*ct))/1e3:.0f} kHz, Av = {c1/c2:.0f}")

Wien (6.2k, 0.02uF): 1284 Hz
Phase-shift (4.7k, 0.022uF): 628 Hz, Rf = 136.3 kOhm
Colpitts: CT = 42.7 pF, f = 770 kHz, Av = 10


**Comment:** 1284 Hz, 628 Hz (as in the book) and Rf = 136.3 kΩ; the Colpitts circuit needs Av = 10 (as in the book).

## 2. Start-up and amplitude (loop-gain model)

In [3]:
# amplitude grows by loop gain each cycle until the limiter saturates it (|v| <= 1)
for g in (0.9, 1.0, 1.05, 1.2):
    v = 1e-3; n = 0
    while abs(v) < 1 and n < 1000: v *= g; n += 1
    print(f"loop gain {g:.2f}: {"dies out" if g < 1 else ("stays at 1 mV" if g == 1 else f"reaches 1 V after {n} cycles")}")

loop gain 0.90: dies out
loop gain 1.00: stays at 1 mV
loop gain 1.05: reaches 1 V after 142 cycles
loop gain 1.20: reaches 1 V after 38 cycles


**Comment:** a loop gain below 1 makes the oscillation die out, exactly 1 holds the amplitude, above 1 makes it grow until limited — in line with Floyd §16–2.

## 3. 555 astable: formula and capacitor charge/discharge simulation

In [4]:
R1, R2, C, Vcc = 1e3, 3.3e3, 0.047e-6, 5.5
f_formula = 1.44/((R1+2*R2)*C); duty = (R1+R2)/(R1+2*R2)
print(f"formula: f = {f_formula:.0f} Hz, duty = {duty*100:.1f} %")
tH = np.log(2)*(R1+R2)*C; tL = np.log(2)*R2*C; print(f"ideal exponentials: tH = {tH*1e6:.1f} us, tL = {tL*1e6:.1f} us -> f = {1/(tH+tL):.0f} Hz")
dt = 1e-8; v = Vcc/3; t = 0; up = True; edges = []
while len(edges) < 8 and t < 5e-3:
    if up:
        v += (Vcc - v)/((R1+R2)*C)*dt
        if v >= 2*Vcc/3: up = False; edges.append(t)
    else:
        v += (0 - v)/(R2*C)*dt
        if v <= Vcc/3: up = True; edges.append(t)
    t += dt
T = (edges[-1]-edges[1])/3
print(f"simulated period = {T*1e6:.1f} us -> f = {1/T:.0f} Hz")

formula: f = 4031 Hz, duty = 56.6 %
ideal exponentials: tH = 140.1 us, tL = 107.5 us -> f = 4039 Hz
simulated period = 247.6 us -> f = 4039 Hz


**Comment:** Floyd’s formula gives about 4.03 kHz; the ideal exponential model (ln 2 = 0.693 instead of 0.7) gives a very close value and the numerical simulation agrees with that model. The small gap between 1.44 and 1/(2·0.693) is only Floyd’s rounding of 0.7.

## 4. Duty cycle and Cext design

In [5]:
print("Cext for 25 kHz:", round(1.44/((1e3+2*3.3e3)*25e3)*1e6, 4), "uF")
print("R2 for 75 % duty with R1 = 3.3k:", 3.3/2, "kOhm")
for r1, r2 in [(1, 1), (1, 10), (10, 1)]: print(f"R1={r1}k R2={r2}k duty = {(r1+r2)/(r1+2*r2)*100:.1f} %")

Cext for 25 kHz: 0.0076 uF
R2 for 75 % duty with R1 = 3.3k: 1.65 kOhm
R1=1k R2=1k duty = 66.7 %
R1=1k R2=10k duty = 52.4 %
R1=10k R2=1k duty = 91.7 %


**Comment:** Cext = 0.0076 μF (as in the book); R2 = 1.65 kΩ gives 75 %; as R2 ≪ R1 the duty approaches 100 %, as R2 ≫ R1 it approaches 50 % — never below 50 % without the diode.